# Week 01 - Lab 03  
Today We will build something with immediate value!


Copy your Resume or document in the same folder as of this notebook is.

## Looking up Packages  
In this lab, we're going to use wonderful Gradio package for building UIs, and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking ChatGPT or Claude, and you find all open-source packages on the repository `https://pypi.org`

In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [ ]:
load_dotenv(override = True)
openai = OpenAI()

In [ ]:
reader = PdfReader("./Fayyaz_Ahmad_AI.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [ ]:
print(resume)

In [ ]:
with open('./cover_letter.txt', 'r', encoding = 'utf-8') as f:
    summary = f.read()

In [ ]:
name = "Fayyaz ahmad"

In [ ]:
system_prompt = f"""You are acting as {name}. You are answering questions on {name}'s Website, \
    particularly questions related to {name}'s career, background, skills and experience. \
    Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
    You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
    Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
    If you don't know the answer, say so.
    "
system_prompt += f"\n\n## Summary: \n{summary}\n\n## LinkedIn Profile: \n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}"""

In [ ]:
system_prompt

In [ ]:
def chat(message, history):
    messages = [
        {'role': 'system', 'content': system_prompt}] + history + [{'role': 'user', 'content': message}]
    response = openai.chat.completions.create(
        model = 'gpt-4o-mini',
        messages = messages,
    )
    return response.choices[0].message.content

### Special Note if you are not using OpenAI  
Some Providers, like Groq, might give error when you send your second message in the chat.  
This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.  
If this happens, the solution is to add this first line to the chat() fuction above. It cleans up the history variable.

```python
history = [{'role' : h['role'], 'content' : h['content']} for h in history]
```  
You may need to add this in other chat() calback functions in the future, too.

In [ ]:
gr.ChatInterface(chat).launch()

### **A lot is about to happen...**  
1. Be able to ask an LLM to evaluate an answer.
2. Be able to rerun if the answer fails evaluation.
3. Put this together inot 1 workflow.  

All without any Agentic framework!

In [ ]:
# Create a Pydantic model for the Evaluation
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [ ]:
evaluation_system_prompt = f"""
You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their resume and summary text. Here's the information: 
"""
evaluation_system_prompt  += f"\n\n## Summary: \n{summary}\n\n## Resume: \n {resume}\n\n"
evaluation_system_prompt += "With this context, please evaluate the latest response, replying with whethr the response is acceptable and your feedback."


In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User : \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += 'Please evaluate the response, replying with whether it is acceptable and your feedback.'

    return user_prompt

In [ ]:
import os
ollama = OpenAI(api_key = 'ollama', base_url = 'http://localhost:11434/v1/')


In [ ]:
def evaluate(reply, message, history) -> Evaluation:
    message = [{'role': 'system', 'content': evaluation_system_prompt}]+ [{'role': 'user', 'content': evaluator_user_prompt}]
    response = ollama.chat.completions.parse(
        model = 'gemma4:e4b', 
        messages = messages,
        response_format = Evaluation
    )
    return response.choices[0].message.parsed

In [ ]:
messages = [{'role': 'system', 'content': system_prompt}] + [{'role': 'user', 'content': 'do you hold any appreciation.'}]
response = ollama.chat.completions.create(
    model = 'gemma4:e4b',
    messages = messages
)

reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, 'do you have any accomplishment?', messages[:1])

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer: \n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection: \n{feedback}\n\n"
    messages = [{
        'role': 'system', 'content' : updated_system_prompt
    }] + history + [{'role' : 'user', 'content' : message}]
    response = ollama.chat.completions.create(
        model = 'gemma4:e4b',
        messages = messages
    )
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if 'accomplishment' in message:
        system = system_prompt + "\n\nEverythin in your reply needs to be in a pig latin - \
            it is mandatory that you respond only and entirely in pig lating"
    else:
        system = system_prompt
    
    messages = [{'role': 'system', 'content': system}] + history + [{'role': 'user', 'content': message}]
    response = ollama.chat.completions.create(
        model = 'gemma4:e4b',
        messages = messages,

    )
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [ ]:
gr.ChatInterface(chat).launch()